# S3.23 - Notebook exit values
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** A child notebook reports back to its parent with dbutils.notebook.exit(string)

In [0]:
# ============================================================
# S3 close-out: complete code with validation (one cell)
#
# Goal:
# 1. S3.23: create the child, run mode=ok and mode=fail, validate
#    what the parent receives
# 2. S3.20 / S3.21: validate what Spark does with each read mode,
#    _corrupt_record and badRecordsPath, including row 4 (extra token)
# 3. Print a PASS / FAIL / INFO summary table
#
# Nothing here calls exit(), so the cell always reaches the summary.
# ============================================================

import base64, json
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat, Language

checks = []   # (name, status, detail)

def record(name, ok, detail=""):
    checks.append((name, "PASS" if ok else "FAIL", detail))

def info(name, detail):
    checks.append((name, "INFO", detail))

# ------------------------------------------------------------
# PART A - S3.23 parent / child
# ------------------------------------------------------------

# Step A1 - child path, next to this notebook
try:
    from dbruntime.databricks_repl_context import get_context
    me = get_context().notebookPath
except Exception:
    me = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
child = me.rsplit("/", 1)[0] + "/S3.23_child_notebook"
print("Child path:", child)

# Step A2 - child source: ok returns JSON, fail raises
child_src = '''import json
dbutils.widgets.text("mode", "ok")
mode = dbutils.widgets.get("mode").strip().lower()
if mode not in {"ok", "fail"}:
    raise ValueError(f"Invalid mode: {mode}")
if mode == "fail":
    raise ValueError("simulated failure")
n = spark.table("samples.tpch.orders").count()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "rows": n}))
'''

try:
    WorkspaceClient().workspace.import_(
        path=child,
        content=base64.b64encode(child_src.encode()).decode(),
        format=ImportFormat.SOURCE, language=Language.PYTHON, overwrite=True)
    record("A1 child notebook created", True)
except Exception as e:
    record("A1 child notebook created", False,
           f"{type(e).__name__}: {str(e).splitlines()[0][:120]}")

# Step A3 - mode=ok: the parent must receive a JSON string
expected_rows = spark.table("samples.tpch.orders").count()
try:
    r = dbutils.notebook.run(child, 300, {"mode": "ok"})
    d = json.loads(r)
    record("A2 run(ok) returns a str", isinstance(r, str), f"type={type(r).__name__}")
    record("A3 status is SUCCESS", d.get("status") == "SUCCESS", f"status={d.get('status')}")
    record("A4 rows match the parent count", d.get("rows") == expected_rows,
           f"child={d.get('rows')} parent={expected_rows}")
except Exception as e:
    record("A2 run(ok) returns a str", False,
           f"{type(e).__name__}: {str(e).splitlines()[0][:120]}")

# Step A4 - mode=fail: the child failure must reach the parent as an exception
try:
    dbutils.notebook.run(child, 300, {"mode": "fail"})
    record("A5 run(fail) raises in the parent", False, "returned normally")
except Exception as e:
    record("A5 run(fail) raises in the parent", True, type(e).__name__)

# ------------------------------------------------------------
# PART B - S3.20 / S3.21 read modes and bad records
# ------------------------------------------------------------

# Step B1 - the same 5-row CSV used in S3.20 and S3.21
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.s3_21_files")
base = "/Volumes/workspace/default/s3_21_files"
csv, bad = f"{base}/row4_check.csv", f"{base}/row4_bad"
dbutils.fs.rm(bad, True)
dbutils.fs.put(csv, "\n".join(["id,city,qty", "1,Delhi,10", "2,Mumbai,20",
               "3,Pune,abc", "4,Chennai,40,EXTRA", "5,Delhi,50"]), True)
schema = "id INT, city STRING, qty INT"

def ids(df):
    return sorted(r["id"] for r in df.collect() if r["id"] is not None)

# Step B2 - each read mode (FAILFAST must raise on row 3)
for mode in ["PERMISSIVE", "DROPMALFORMED"]:
    df = spark.read.option("header", "true").option("mode", mode).schema(schema).csv(csv)
    got = ids(df)
    info(f"B1 {mode} returned ids", str(got))
    if mode == "DROPMALFORMED":
        record("B2 DROPMALFORMED drops row 3", 3 not in got, f"ids={got}")

try:
    spark.read.option("header", "true").option("mode", "FAILFAST") \
         .schema(schema).csv(csv).collect()
    record("B3 FAILFAST raises", False, "no exception")
except Exception as e:
    record("B3 FAILFAST raises", True, type(e).__name__)

# Step B3 - _corrupt_record: which rows keep their raw line?
df = spark.read.option("header", "true").option("mode", "PERMISSIVE") \
          .schema(schema + ", _corrupt_record STRING").csv(csv)
corrupt = [(r["id"], r["_corrupt_record"]) for r in df.collect()
           if r["_corrupt_record"] is not None]
info("B4 rows with _corrupt_record", str(corrupt) if corrupt else "none")

# Step B4 - badRecordsPath: returned ids and every captured record
df = spark.read.option("header", "true").option("badRecordsPath", bad).schema(schema).csv(csv)
returned = ids(df)
info("B5 badRecordsPath returned ids", str(returned))

def files(p):
    out = []
    for i in dbutils.fs.ls(p):
        out += files(i.path) if i.path.endswith("/") else [i.path]
    return out

captured = []
try:
    for f in files(bad):
        if not f.rsplit("/", 1)[-1].startswith("_"):
            captured += [r["value"] for r in spark.read.text(f).collect()]
except Exception as e:
    info("B6 listing bad-record path", f"{type(e).__name__}")

record("B7 row 3 captured in the bad-record file",
       any("3,Pune,abc" in c for c in captured), f"{len(captured)} line(s)")
info("B8 row 4 captured in the bad-record file", str(any("EXTRA" in c for c in captured)))
info("B9 row 4 returned by badRecordsPath read", str(4 in returned))

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------
print("\n================ S3 CLOSE-OUT SUMMARY ================")
for name, status, detail in checks:
    print(f"{status:<5} | {name:<45} | {detail}")
failed = [c for c in checks if c[1] == "FAIL"]
print(f"\nPASS: {sum(c[1]=='PASS' for c in checks)}  FAIL: {len(failed)}  INFO: {sum(c[1]=='INFO' for c in checks)}")

# S3.23 — Key Takeaways: Notebook Exit Values

## 1. exit() and run()

The child notebook can return a value using:

```python
dbutils.notebook.exit(value)
```

The parent receives that value from:

```python
dbutils.notebook.run(...)
```

`dbutils.notebook.exit()` takes one string value, so when several logical values must be returned, encode them as JSON:

```python
result = {"status": "SUCCESS", "rows": row_count}
dbutils.notebook.exit(json.dumps(result))
```

The parent decodes the returned string with `json.loads(...)`.

---

## 2. Parameters

The parent passes parameters through `dbutils.notebook.run()`:

```python
dbutils.notebook.run(child_path, 300, {"mode": "ok"})
```

The child receives them through a widget:

```python
dbutils.widgets.text("mode", "ok")
mode = dbutils.widgets.get("mode")
```

```text
Parent arguments -> Child widgets -> Child logic
```

---

## 3. Success Path (mode = ok)

```text
read samples.tpch.orders -> count rows -> build JSON result -> dbutils.notebook.exit(...)
```

---

## 4. Failure Path (mode = fail)

```python
if mode == "fail":
    raise ValueError("simulated failure")
```

A real failure should be raised, not disguised as a successful exit value.

```text
mode = fail -> raise ValueError -> child run fails -> parent run() raises
```

---

## 5. exit() vs raise

| | Meaning | Parent sees |
|---|---|---|
| exit() | Notebook completed normally | A returned string |
| raise | Execution failed | An exception |

---

## 6. Do Not Return Large Data Through exit()

Use `exit()` for small control information: status, row count, table name, path, run ID, small JSON.

For large results, persist the data and return the table name or path.

---

## 7. Observed in Our Workspace (verified, Spark 4.2.0, serverless)

Parent notebook, close-out run (PASS 8, FAIL 0):

```text
dbutils.notebook.run(mode=ok)   -> returned type: str
                                   status = SUCCESS
                                   rows   = 7500000 (equal to the parent's own count)
dbutils.notebook.run(mode=fail) -> raised Py4JJavaError in the parent
```

Notebook Workflows view:

```text
mode = ok    Succeeded (29 s)
mode = fail  Failed, error code RunExecutionError (9 s)
```

Setup issue met on the way:

- The first parent run() failed with "Unable to access the notebook" because the child notebook was outside the repo folder. A search of the repo confirmed it was not there.
- Creating the child in the same folder as the parent fixed it.

---

## 8. Not Covered Here

- Task dependencies, retries, scheduling and job-level monitoring are covered in Jobs and Workflows (S11).

---

## 9. Checkpoint

**Q1. How does a child return information to a parent?**
Child: `dbutils.notebook.exit(...)`. Parent: receives it from `dbutils.notebook.run(...)`.

**Q2. Why use `json.dumps()`?**
`exit()` takes one string. JSON encodes several fields (status, rows, message) into that one string.

**Q3. How are parameters passed to the child?**
The parent supplies arguments to `run()`. The child reads them through widgets.

**Q4. What should happen on a real execution failure?**
Raise an exception. Do not return a fake SUCCESS value.

**Q5. What happened in our mode="fail" test?**
The parent's `run()` raised Py4JJavaError, and the Workflows view showed Failed with RunExecutionError.

**Q6. What happened in our mode="ok" test?**
`run()` returned a `str`. Decoded: status SUCCESS, rows 7500000. The Workflows view showed Succeeded.

---

## 10. Core Understanding

```text
Pass input to child?          -> arguments + widgets
Normal child result?          -> dbutils.notebook.exit()
Several result values?        -> json.dumps()
Real execution failure?       -> raise an exception
Parent / child orchestration? -> dbutils.notebook.run()
Production orchestration?     -> Jobs and Workflows (S11)
```

**S3.23 — Notebook Exit Values completed and verified.**